# Deep Sort + Yolo and Reid

In [ ]:
from google.colab import files
files.upload()
!unzip -q -o deepsort-mod.zip -d /content
%cd /content/deepsort-mod

In [ ]:
!pip install -q tensorflow gdown torch torchvision ultralytics torchreid
!pip install -q git+https://github.com/JonathonLuiten/TrackEval.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive"
!unzip -q -o "{DRIVE_DIR}/sequences.zip" -d data

In [ ]:
import os

sequences = ["TUD-Campus", "TUD-Stadtmitte", "KITTI-17", "PETS09-S2L1", "MOT16-09", "MOT16-11"]
for name in sequences:
    d = f"data/sequences/{name}"
    have = {p: os.path.exists(f"{d}/{p}") for p in ["img1", "det/det.txt", "gt/gt.txt", "seqinfo.ini"]}
    print(name, have)

## Original Deep SORT

In [ ]:
!mkdir -p resources/networks
!gdown -q 1bB66hP9voDXuoBoaCcKYY7a8IYzMMs4P -O resources/networks/mars-small128.pb

In [ ]:
!python tools/generate_detections.py \
    --model=resources/networks/mars-small128.pb \
    --mot_dir=data/sequences \
    --output_dir=data/detections

In [ ]:
!python evaluate_motchallenge.py \
    --mot_dir=data/sequences \
    --detection_dir=data/detections \
    --output_dir=results/baseline \
    --min_confidence=0.3 \
    --nn_budget=100

In [ ]:
!python eval_hota.py --results_dir=results/baseline

## YOLO + OSNet REID

In [ ]:
!python run_tracker.py \
    --mot_dir=data/sequences \
    --output_dir=results/yolo_osnet \
    --detector=yolov8n \
    --reid=osnet \
    --min_confidence=0.3 \
    --nn_budget=100

## Best configuration (tuned)

In [ ]:
!python run_tracker.py \
    --mot_dir=data/sequences \
    --output_dir=results/best \
    --detector=yolov8s \
    --reid=osnet_x1_0 \
    --min_confidence=0.5 \
    --max_cosine_distance=0.3 \
    --max_age=60

## Identity database (additional task)

In [ ]:
!python run_identity.py \
    --mot_dir=data/sequences \
    --output_dir=results/identity \
    --detector=yolov8s \
    --reid=osnet_x1_0 \
    --min_confidence=0.5 \
    --max_cosine_distance=0.3 \
    --max_age=60 \
    --threshold=0.35 \
    --window=180

## Segmentation

In [ ]:
!python run_tracker.py \
    --mot_dir=data/sequences \
    --output_dir=results/seg \
    --detector=yolov8s-seg \
    --reid=osnet_x1_0 \
    --min_confidence=0.5 \
    --max_cosine_distance=0.3 \
    --max_age=60 \
    --mask